# Model Datavidia 11.0 Task 1: CatBoost GPU (Windows + RTX 5060)

Notebook ini dijalankan secara lokal di Windows, menggunakan GPU NVIDIA jika tersedia. Feature engineering dan alur training tetap dipertahankan.


**Deadline configuration:** Benchmark disabled by default; training uses 400 iterations, depth 6, 2-fold CV, early stopping 50. Run cells sequentially from top.


## 0. Setup

Struktur dataset lokal yang diharapkan:
```text
Datavidia-2026/
└── dataset/
    └── Task-1-Datavidia-11.0/
        ├── train/  (CSV train)
        └── test/   (CSV test)
```
Pastikan working directory notebook berada di folder project `Datavidia-2026`, atau ubah `DATASET_RELATIVE`/`BASE` di cell setup.


In [10]:



import numpy as np
import pandas as pd
import gc, os, subprocess, time, sys
try:
    import psutil
except ImportError:
    psutil = None


def log_memory(label):
    if psutil is not None:
        ram = psutil.Process(os.getpid()).memory_info().rss / 1024**3
        print(f"[{label}] Process RAM: {ram:.2f} GB")
    else:
        print(f"[{label}] Install psutil for RAM monitoring: %pip install psutil")

from pathlib import Path
from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report
from scipy.optimize import minimize

pd.set_option("display.max_columns", 100)


DATASET_RELATIVE = Path("dataset/Task-1-Datavidia-11.0")
base_candidates = [
    Path.cwd() / DATASET_RELATIVE,
    Path.cwd().parent / DATASET_RELATIVE,
    Path(r"C:\Users\LENOVO\DataspellProjects\Datavidia-2026") / DATASET_RELATIVE,
]

BASE = next((p for p in base_candidates if (p / "train").is_dir() and (p / "test").is_dir()), None)

if BASE is None:
    checked = "\\n".join(str(p) for p in base_candidates)
    raise FileNotFoundError(
        "Folder dataset tidak ditemukan. Pastikan folder dataset berisi train dan test.\\n"
        f"Lokasi yang diperiksa:\\n{checked}\\n"
        "Atau atur BASE secara manual."
    )

BASE = BASE.resolve()
TRAIN_DIR = BASE / "train"
TEST_DIR = BASE / "test"
TARGET = "status_penanganan"

print("Dataset root :", BASE)
print("Train folder :", TRAIN_DIR)
print("Test folder  :", TEST_DIR)
print("Train CSV    :", sorted(p.name for p in TRAIN_DIR.glob("*.csv")))
print("Test CSV     :", sorted(p.name for p in TEST_DIR.glob("*.csv")))

required_files = [
    "shipments.csv", "couriers.csv", "customers.csv", "hubs.csv",
    "promotions.csv", "route_segments.csv", "scan_events.csv",
    "service_packages.csv",
]
for split_dir in (TRAIN_DIR, TEST_DIR):
    missing = [name for name in required_files if not (split_dir / name).is_file()]
    if missing:
        raise FileNotFoundError(f"File berikut tidak ditemukan di {split_dir}: {missing}")


try:
    smi_output = subprocess.check_output(["nvidia-smi"], text=True, encoding="utf-8", errors="replace")
    print(smi_output[:2500])
    GPU_AVAILABLE = True
except Exception as e:
    print("nvidia-smi tidak berhasil dijalankan:", repr(e))
    GPU_AVAILABLE = False

TASK_TYPE = "GPU" if GPU_AVAILABLE else "CPU"
print("CatBoost task_type:", TASK_TYPE)


Dataset root : C:\Users\LENOVO\DataspellProjects\Datavidia-2026\dataset\Task-1-Datavidia-11.0
Train folder : C:\Users\LENOVO\DataspellProjects\Datavidia-2026\dataset\Task-1-Datavidia-11.0\train
Test folder  : C:\Users\LENOVO\DataspellProjects\Datavidia-2026\dataset\Task-1-Datavidia-11.0\test
Train CSV    : ['couriers.csv', 'customers.csv', 'hubs.csv', 'invoices.csv', 'promotions.csv', 'route_segments.csv', 'scan_events.csv', 'service_packages.csv', 'shipments.csv']
Test CSV     : ['couriers.csv', 'customers.csv', 'hubs.csv', 'invoices.csv', 'promotions.csv', 'route_segments.csv', 'scan_events.csv', 'service_packages.csv', 'shipments.csv']
Wed Nov 25 22:01:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 592.15                 Driver Version: 592.15         CUDA Version: 13.1     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | B

## 1. Feature engineering

In [11]:
def load_all(split):
    folder = TRAIN_DIR if split == "train" else TEST_DIR

    
    
    
    names = [
        "shipments", "scan_events", "route_segments",
        "customers", "couriers", "hubs",
        "service_packages", "promotions"
    ]

    tables = {}
    for name in names:
        csv_path = folder / f"{name}.csv"
        if not csv_path.is_file():
            raise FileNotFoundError(f"File wajib tidak ditemukan: {csv_path}")

        if name == "scan_events":
            tables[name] = pd.read_csv(
                csv_path,
                low_memory=False,
                dtype={"event_type": "category", "device_type": "category"}
            )
        else:
            tables[name] = pd.read_csv(csv_path, low_memory=False)

    return tables


def agg_scan(ev_raw):
    ev = ev_raw.copy()

    
    is_selesai = ev["event_type"] == "selesai"
    has_selesai = (
        is_selesai.groupby(ev["shipment_id"]).any()
        .astype("int8").rename("has_selesai")
    )

    
    ev = ev.loc[~is_selesai].copy()
    ev["gap_gt_24"] = (ev["gap_hours"] > 24).astype("int8")
    ev["gap_gt_48"] = (ev["gap_hours"] > 48).astype("int8")

    g = ev.groupby("shipment_id", sort=False)
    out = g.agg(
        n_scan=("gap_hours", "size"),
        gap_sum=("gap_hours", "sum"),
        gap_mean=("gap_hours", "mean"),
        gap_max=("gap_hours", "max"),
        gap_std=("gap_hours", "std"),                       
        manual_ratio=("is_manual", "mean"),
        q_mean=("scan_quality", "mean"),
        q_min=("scan_quality", "min"),
        q_std=("scan_quality", "std"),                      
        n_gap_gt_24=("gap_gt_24", "sum"),
        n_gap_gt_48=("gap_gt_48", "sum"),
        n_unique_ev=("event_type", "nunique"),              
    )

    counts = (
        ev.groupby(["shipment_id", "event_type"], observed=True)
        .size().unstack(fill_value=0).add_prefix("ev_")
    )
    out = out.join(counts)

    
    upaya_col = "ev_upaya_antar"
    pickup_col = "ev_pickup"
    out["n_upaya"] = out[upaya_col].clip(upper=6) if upaya_col in out else 0
    out["n_pickup"] = out[pickup_col].clip(upper=5) if pickup_col in out else 0
    out["gap_cv"] = out["gap_std"] / (out["gap_mean"].abs() + 1e-3)   

    
    last_ev = (
        ev.groupby("shipment_id", sort=False)["event_type"]
        .last().astype(str).rename("last_event")                       
    )
    out = out.join(last_ev)

    
    out = out.join(has_selesai)                                        

    return out


def agg_route(rt):
    rt = rt.copy()
    rt["dur_kosong"] = rt["durasi_jam"].isna().astype("int8")

    g = rt.groupby("shipment_id", sort=False)

    out = g.agg(
        n_seg=("urutan", "size"),
        jarak_seg=("jarak_segmen_km", "sum"),
        durasi_seg=("durasi_jam", "sum"),
        dur_kosong=("dur_kosong", "sum"),
        biaya_seg=("biaya_segmen", "sum"),
        n_hub_seg=("segmen_hub_id", "nunique"),
    )

    out["kecepatan"] = (
        out["jarak_seg"] / out["durasi_seg"].replace(0, np.nan)
    )
    out["biaya_per_km"] = (
        out["biaya_seg"] / out["jarak_seg"].replace(0, np.nan)
    )

    moda = (
        rt.groupby(["shipment_id", "moda"], observed=True)
        .size()
        .unstack(fill_value=0)
        .add_prefix("moda_")
    )

    out = out.join(moda)
    out["n_moda"] = (moda > 0).sum(axis=1)

    return out


def join_master(shp, tbl, key, prefix):
    
    if tbl is None or key not in tbl.columns or key not in shp.columns:
        return shp

    sub = tbl.set_index(key).add_prefix(prefix)
    sub = sub.loc[:, ~sub.columns.isin(shp.columns)]
    return shp.join(sub, on=key, validate="m:1")


def add_derived_features(shp):
    shp["off_hours"] = (
        (shp["ship_hour"] < 6) | (shp["ship_hour"] >= 20)
    ).astype("int8")

    shp["has_note"] = (shp["jumlah_catatan"] > 0).astype("int8")
    shp["peak_month"] = shp["ship_month"].isin([3, 10, 11, 12]).astype("int8")
    shp["log_declared"] = np.log1p(shp["declared_value"])

    if {"weight_kg", "kr_kapasitas_muatan_kg"}.issubset(shp.columns):
        shp["lewat_kapasitas"] = (
            shp["weight_kg"] > shp["kr_kapasitas_muatan_kg"]
        ).astype("int8")

    if {"weight_kg", "sv_max_weight_kg"}.issubset(shp.columns):
        shp["lewat_batas_paket"] = (
            shp["weight_kg"] > shp["sv_max_weight_kg"]
        ).astype("int8")

    if {"is_cod", "sv_cod_allowed"}.issubset(shp.columns):
        shp["cod_dilarang"] = (
            (shp["is_cod"] == 1) & (shp["sv_cod_allowed"] == 0)
        ).astype("int8")

    if {"hub_id", "kr_hub_id"}.issubset(shp.columns):
        shp["beda_hub_kurir"] = (
            shp["hub_id"] != shp["kr_hub_id"]
        ).astype("int8")

    if {"ship_hour", "hb_jam_buka", "hb_jam_tutup"}.issubset(shp.columns):
        shp["luar_jam_hub"] = (
            (shp["ship_hour"] < shp["hb_jam_buka"]) |
            (shp["ship_hour"] >= shp["hb_jam_tutup"])
        ).astype("int8")

    promo_cols = {"pr_bulan_mulai", "pr_bulan_akhir", "ship_month"}
    if promo_cols.issubset(shp.columns):
        start = shp["pr_bulan_mulai"]
        end = shp["pr_bulan_akhir"]
        month = shp["ship_month"]

        active = np.where(
            start <= end,
            (month >= start) & (month <= end),
            (month >= start) | (month <= end)
        )

        shp["promo_aktif"] = np.where(start.isna(), -1, active).astype("int8")

    return shp


def build_features(split):
    t0 = time.perf_counter()
    print(f"\n--- Build features: {split} ---")
    tables = load_all(split)
    log_memory(f"{split}: CSV loaded")
    shp = tables["shipments"]  

    if "scan_events" in tables:
        scan = agg_scan(tables["scan_events"])
        shp = shp.merge(scan, left_on="shipment_id", right_index=True, how="left")
        del scan, tables["scan_events"]
        gc.collect()
        log_memory(f"{split}: scan aggregated")

    if "route_segments" in tables:
        route = agg_route(tables["route_segments"])
        shp = shp.merge(route, left_on="shipment_id", right_index=True, how="left")
        del route, tables["route_segments"]
        gc.collect()
        log_memory(f"{split}: route aggregated")

    shp["has_scan"] = shp["n_scan"].notna().astype("int8") if "n_scan" in shp else 0
    shp["has_route"] = shp["n_seg"].notna().astype("int8") if "n_seg" in shp else 0

    shp = join_master(shp, tables.get("couriers"), "courier_id", "kr_")
    shp = join_master(shp, tables.get("customers"), "customer_id", "cu_")
    shp = join_master(shp, tables.get("hubs"), "hub_id", "hb_")
    shp = join_master(shp, tables.get("service_packages"), "package_id", "sv_")
    shp = join_master(shp, tables.get("promotions"), "promo_id", "pr_")
    del tables
    gc.collect()

    shp = add_derived_features(shp)
    print(f"{split} feature rows/cols: {shp.shape}; elapsed: {(time.perf_counter()-t0)/60:.1f} min")
    log_memory(f"{split}: features finished")

    drop_anon = [
        "skor_prioritas_kirim", "indeks_rute_sekunder",
        "faktor_muatan_internal", "varian_cuaca_rute",
        "indeks_waktu_proses", "skor_kualitas_cek"
    ]

    shp = shp.drop(columns=[c for c in drop_anon if c in shp.columns])

    return shp

Agregasi invoice tidak digunakan karena `invoice_type`, `payment_status`, `is_penalty`, dan `days_to_pay` berpotensi terbentuk setelah status penanganan diketahui. Fitur scan terminal `selesai` juga dikeluarkan dari agregasi untuk mengurangi risiko kebocoran target.

## 2. Membuat fitur train dan test

In [12]:
feature_start = time.perf_counter()
df_train = build_features("train")
df_test = build_features("test")
log_memory("Both feature frames in memory")

hub_month = (
    df_train.groupby(["hub_id", "ship_month"], dropna=False)
    .size()
    .rename("kirim_hub_bulan")
)

for frame in (df_train, df_test):
    keys = pd.MultiIndex.from_frame(frame[["hub_id", "ship_month"]])
    frame["kirim_hub_bulan"] = hub_month.reindex(keys).fillna(0).to_numpy()

_, density_bins = pd.qcut(
    df_train["kirim_hub_bulan"],
    q=3,
    duplicates="drop",
    retbins=True
)

density_bins[0] = -np.inf
density_bins[-1] = np.inf

for frame in (df_train, df_test):
    frame["kepadatan"] = pd.cut(
        frame["kirim_hub_bulan"],
        bins=density_bins,
        include_lowest=True
    ).astype(str)

df_train.to_parquet("features_train.parquet", index=False)
df_test.to_parquet("features_test.parquet", index=False)

print("Train:", df_train.shape)
print("Test :", df_test.shape)
print(f"Total feature engineering: {(time.perf_counter()-feature_start)/60:.1f} minutes")

del df_train, df_test, hub_month
gc.collect()


--- Build features: train ---
[train: CSV loaded] Process RAM: 4.36 GB
[train: scan aggregated] Process RAM: 4.51 GB
[train: route aggregated] Process RAM: 4.19 GB
train feature rows/cols: (1243617, 130); elapsed: 0.3 min
[train: features finished] Process RAM: 4.40 GB

--- Build features: test ---
[test: CSV loaded] Process RAM: 4.81 GB
[test: scan aggregated] Process RAM: 4.86 GB
[test: route aggregated] Process RAM: 4.72 GB
test feature rows/cols: (362245, 129); elapsed: 0.1 min
[test: features finished] Process RAM: 4.79 GB
[Both feature frames in memory] Process RAM: 4.77 GB
Train: (1243617, 126)
Test : (362245, 125)
Total feature engineering: 0.5 minutes


0

## 3. Persiapan data

CatBoost menerima fitur kategorikal secara native. Kolom kategorikal diubah menjadi string dan nilai kosong diberi token khusus; tidak dilakukan one-hot encoding manual.

In [13]:
prep_start = time.perf_counter()
tr = pd.read_parquet("features_train.parquet")
te = pd.read_parquet("features_test.parquet")
log_memory("Parquet loaded")

y = tr.pop(TARGET).astype(str).values
test_ids = te["shipment_id"].values

tr = tr.drop(columns=["shipment_id"], errors="ignore")
te = te.drop(columns=["shipment_id"], errors="ignore")
te = te.reindex(columns=tr.columns, fill_value=np.nan)

count_cols = [
    c for c in tr.columns
    if c.startswith(("ev_", "moda_"))
]

tr[count_cols] = tr[count_cols].fillna(0)
te[count_cols] = te[count_cols].fillna(0)

cat_cols = tr.select_dtypes(include=["object", "category", "str"]).columns.tolist()

for c in cat_cols:
    tr[c] = tr[c].astype("string").fillna("__MISSING__").astype(str)
    te[c] = te[c].astype("string").fillna("__MISSING__").astype(str)

classes, y_enc = np.unique(y, return_inverse=True)

print(f"Data train: {tr.shape}")
print(f"Data test : {te.shape}")
print(f"Jumlah fitur kategorikal: {len(cat_cols)}")
print(f"Jumlah kelas: {len(classes)}")
print(f"Prep time: {(time.perf_counter()-prep_start)/60:.1f} min")
log_memory("Pool data prepared")

[Parquet loaded] Process RAM: 5.63 GB
Data train: (1243617, 124)
Data test : (362245, 124)
Jumlah fitur kategorikal: 29
Jumlah kelas: 7
Prep time: 0.0 min
[Pool data prepared] Process RAM: 5.63 GB


## 4. Training CatBoost\nUntuk dataset besar, default awal Depth 6 dan VRAM allocation 0.70 lebih konservatif pada RTX 5060 8 GB. Setelah benchmark dan run stabil, depth 7 bisa diuji. Training OOF menggunakan 2 fold.\n

### Benchmark singkat sebelum full training\nJalankan cell berikut untuk mengukur throughput pada subset (default 50k rows / 100 iterasi). Hasilnya memperkirakan durasi full 2-fold; benchmark bersifat opsional.

In [14]:


RUN_BENCHMARK = False


In [15]:

N_SPLITS      = 5       
SEEDS         = [42]    
ITERATIONS    = 2000    
DEPTH         = 7       
EARLY_STOPPING = 100    

def make_params(seed):
    params = dict(
        loss_function="MultiClass",
        eval_metric="MultiClass",
        iterations=ITERATIONS,
        learning_rate=0.03,          
        depth=DEPTH,
        l2_leaf_reg=3,               
        
        random_seed=seed,
        thread_count=-1,
        task_type=TASK_TYPE,
        verbose=100,
        allow_writing_files=False,
        one_hot_max_size=10,         
        max_ctr_complexity=3,        
    )
    if TASK_TYPE == "GPU":
        params["devices"] = "0"
        params["gpu_ram_part"] = 0.70
    return params

oof_avg  = np.zeros((len(tr), len(classes)), dtype=np.float32)
test_avg = np.zeros((len(te), len(classes)), dtype=np.float32)
feature_importance_accum = np.zeros(len(tr.columns), dtype=np.float64)   

test_pool = Pool(te, cat_features=cat_cols)

for seed in SEEDS:
    print(f"Seed: {seed}")
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    oof        = np.zeros((len(tr), len(classes)), dtype=np.float32)
    test_probs = np.zeros((len(te), len(classes)), dtype=np.float32)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(tr, y_enc), start=1):
        print(f"Fold {fold}/{N_SPLITS}: train={len(tr_idx):,}, valid={len(va_idx):,}")
        train_pool = Pool(tr.iloc[tr_idx], label=y_enc[tr_idx], cat_features=cat_cols)
        valid_pool = Pool(tr.iloc[va_idx], label=y_enc[va_idx], cat_features=cat_cols)

        model = CatBoostClassifier(**make_params(seed))
        fit_start = time.perf_counter()
        model.fit(
            train_pool,
            eval_set=valid_pool,
            early_stopping_rounds=EARLY_STOPPING,
            use_best_model=True
        )
        print(f"Fold training time: {(time.perf_counter()-fit_start)/60:.1f} min")

        oof[va_idx] = model.predict_proba(valid_pool)
        test_probs += model.predict_proba(test_pool) / N_SPLITS

        
        feature_importance_accum += model.get_feature_importance() / (N_SPLITS * len(SEEDS))

        fold_f1 = f1_score(y_enc[va_idx], oof[va_idx].argmax(1), average="macro")
        print(f"Fold {fold} | Macro F1: {fold_f1:.4f} | Best iter: {model.get_best_iteration()}")

        del train_pool, valid_pool, model
        gc.collect()

    oof_avg  += oof / len(SEEDS)
    test_avg += test_probs / len(SEEDS)

Seed: 42
Fold 1/5: train=994,893, valid=248,724
0:	learn: 1.8967430	test: 1.8964739	best: 1.8964739 (0)	total: 46.5ms	remaining: 1m 32s
100:	learn: 1.1502598	test: 1.1349916	best: 1.1349916 (100)	total: 4.08s	remaining: 1m 16s
200:	learn: 1.0758215	test: 1.0584883	best: 1.0584883 (200)	total: 7.66s	remaining: 1m 8s
300:	learn: 1.0476964	test: 1.0309848	best: 1.0309848 (300)	total: 11.3s	remaining: 1m 4s
400:	learn: 1.0326827	test: 1.0168911	best: 1.0168911 (400)	total: 15.1s	remaining: 1m
500:	learn: 1.0230419	test: 1.0082214	best: 1.0082214 (500)	total: 18.8s	remaining: 56.3s
600:	learn: 1.0161444	test: 1.0024263	best: 1.0024263 (600)	total: 22.7s	remaining: 52.8s
700:	learn: 1.0106189	test: 0.9981083	best: 0.9981083 (700)	total: 26.4s	remaining: 49s
800:	learn: 1.0057362	test: 0.9938754	best: 0.9938754 (800)	total: 30.2s	remaining: 45.2s
900:	learn: 1.0019787	test: 0.9911638	best: 0.9911638 (900)	total: 34s	remaining: 41.4s
1000:	learn: 0.9984260	test: 0.9889845	best: 0.9889845 (1000

## 5. Evaluasi dan optimasi bobot kelas

In [16]:
def optimize_weights(probs, y_true, n_classes):
    def neg_f1(weights):
        weights = np.clip(weights, 1e-3, 1e3)
        pred = np.argmax(probs * weights, axis=1)
        return -f1_score(y_true, pred, average="macro")
    result = minimize(neg_f1, np.ones(n_classes), method="Nelder-Mead",
                      options={"maxiter": 80})
    return np.clip(result.x, 1e-3, 1e3), -result.fun


baseline_f1 = f1_score(y_enc, oof_avg.argmax(1), average="macro")
print(f"Baseline OOF Macro F1 (no weights): {baseline_f1:.4f}")


rs   = np.random.RandomState(42)
perm = rs.permutation(len(y_enc))
half = len(y_enc) // 2
tune_idx = perm[:half]
eval_idx = perm[half:]

OPT_ROWS = min(400_000, half)
opt_sub  = rs.choice(half, size=OPT_ROWS, replace=False)
opt_idx  = tune_idx[opt_sub]

best_w, _ = optimize_weights(oof_avg[opt_idx], y_enc[opt_idx], len(classes))


held_out_f1 = f1_score(
    y_enc[eval_idx],
    np.argmax(oof_avg[eval_idx] * best_w, axis=1),
    average="macro"
)
print(f"Held-out OOF Macro F1 (unbiased): {held_out_f1:.4f}")
print("Best weights:", np.round(best_w, 3).tolist())


pred_holdout = np.argmax(oof_avg[eval_idx] * best_w, axis=1)
print(classification_report(y_enc[eval_idx], pred_holdout,
                            target_names=classes, digits=4))

Baseline OOF Macro F1 (no weights): 0.5088
Held-out OOF Macro F1 (unbiased): 0.5296
Best weights: [1.125, 1.435, 0.655, 0.576, 1.449, 1.288, 0.848]
                      precision    recall  f1-score   support

    gagal_antar_awal     0.3899    0.3871    0.3885     62167
        klaim_hilang     0.4289    0.4443    0.4365     18778
   konfirmasi_alamat     0.6895    0.6561    0.6724    136517
              normal     0.7715    0.7695    0.7705    236871
      retur_pengirim     0.3578    0.4283    0.3899     43916
    rusak_perjalanan     0.4198    0.3957    0.4074     31065
tertunda_operasional     0.6417    0.6426    0.6421     92495

            accuracy                         0.6349    621809
           macro avg     0.5285    0.5319    0.5296    621809
        weighted avg     0.6389    0.6349    0.6366    621809



## 6. Feature Importance

In [17]:
imp_df = pd.DataFrame({
    "feature":    tr.columns,
    "importance": feature_importance_accum,
}).sort_values("importance", ascending=False).reset_index(drop=True)
imp_df["cum_pct"] = imp_df["importance"].cumsum() / imp_df["importance"].sum() * 100

print("=== TOP 30 fitur ===")
print(imp_df.head(30).to_string(index=False))

print("\n=== BOTTOM 20 fitur (kandidat DROP) ===")
print(imp_df.tail(20).to_string(index=False))

print(f"\nFitur yang memberi 95% total importance: "
      f"{int((imp_df['cum_pct'] <= 95).sum()) + 1} dari {len(imp_df)}")


DROP_CANDIDATES = imp_df.tail(20)["feature"].tolist()
print("\nKandidat drop:", DROP_CANDIDATES)

=== TOP 30 fitur ===
                 feature  importance   cum_pct
        hb_density_index   13.833583 13.833583
              courier_id    7.962853 21.796436
      cu_complaint_index    7.086084 28.882520
       hb_capacity_index    6.519747 35.402266
      cu_address_quality    5.464368 40.866635
         cu_credit_score    5.379272 46.245907
               kecepatan    5.362542 51.608449
                gap_mean    4.849165 56.457615
                  hub_id    4.111474 60.569088
        kr_courier_score    3.738356 64.307444
              durasi_seg    3.288904 67.596348
          declared_value    3.250129 70.846478
              peak_month    3.173944 74.020422
              moda_udara    2.338299 76.358721
                  q_mean    2.222777 78.581497
     hb_automation_level    2.194543 80.776040
             customer_id    2.140413 82.916452
       hb_luas_gudang_m2    1.706312 84.622764
               moda_laut    1.228342 85.851106
               hb_region    1.021765 86

### Cek Fitur Masalah

In [21]:

SUSPECTS = [
    
    "is_promo", "n_hub_seg", "has_scan", "has_route",
    "beda_hub_kurir", "lewat_kapasitas", "cod_dilarang",
    
    "n_upaya", "has_selesai", "last_event", "n_scan",
    "jumlah_catatan", "has_note", "flag_alamat_baru",
    "fragility_flag", "n_gap_gt_48", "n_gap_gt_24",
    "gap_max", "gap_std", "ev_upaya_antar", "ev_pickup",
    "sv_cod_allowed", "sv_max_weight_kg", "kr_hub_id",
]

print("=" * 80)
print("INVESTIGASI FITUR")
print("=" * 80)
for col in SUSPECTS:
    if col not in tr.columns:
        print(f"{col:25s} TIDAK ADA di feature set!")
        continue
    vc = tr[col].value_counts(dropna=False)
    top_val, top_cnt = vc.index[0], vc.iloc[0]
    pct = 100 * top_cnt / len(tr)
    n_unique = tr[col].nunique()
    dtype = tr[col].dtype

    flag = ""
    if pct > 99.0:
        flag = "  KONSTAN (>99%)"
    elif n_unique == 1:
        flag = "  KONSTAN (1 unique)"

    print(f"{col:25s} dtype={str(dtype):10s} unique={n_unique:>7d} "
          f"top={str(top_val)[:15]:15s} ({pct:5.1f}%){flag}")


print("\n" + "=" * 80)
print("EVENT TYPE DISTRIBUTION (raw scan_events train)")
print("=" * 80)
ev_raw = pd.read_csv(TRAIN_DIR / "scan_events.csv",
                     usecols=["shipment_id", "event_type"])
print(ev_raw["event_type"].value_counts().to_string())
print(f"\nShipment tanpa 'selesai': "
      f"{len(set(ev_raw['shipment_id'])) - len(set(ev_raw.loc[ev_raw['event_type']=='selesai', 'shipment_id'])):,}")
print(f"Total shipment unik     : {len(set(ev_raw['shipment_id'])):,}")


print("\n" + "=" * 80)
print("Cek fitur EDA-strong: distribusi per kelas")
print("=" * 80)
for col in ["jumlah_catatan", "flag_alamat_baru", "fragility_flag", "n_upaya", "has_selesai"]:
    if col not in tr.columns:
        continue
    print(f"\n--- {col} ---")
    print(tr.groupby(y_enc)[col].agg(["mean", "std", "median"]).round(4))

del ev_raw
gc.collect()

INVESTIGASI FITUR
is_promo                  dtype=int64      unique=      2 top=0               ( 83.0%)
n_hub_seg                 dtype=int64      unique=      1 top=1               (100.0%)  KONSTAN (>99%)
has_scan                  dtype=int8       unique=      1 top=1               (100.0%)  KONSTAN (>99%)
has_route                 dtype=int8       unique=      1 top=1               (100.0%)  KONSTAN (>99%)
beda_hub_kurir            dtype=int8       unique=      1 top=0               (100.0%)  KONSTAN (>99%)
lewat_kapasitas           dtype=int8       unique=      2 top=0               ( 99.0%)
cod_dilarang              dtype=int8       unique=      1 top=0               (100.0%)  KONSTAN (>99%)
n_upaya                   dtype=int64      unique=      7 top=1               ( 33.9%)
has_selesai               dtype=int8       unique=      2 top=1               ( 66.8%)
last_event                dtype=str        unique=      5 top=upaya_antar     ( 54.0%)
n_scan                    dtype=

0

### Investigasi Fitur #1

#### Ringkasan karakteristik fitur

| Fitur | Tipe data | Jumlah nilai unik | Nilai terbanyak | Persentase | Keterangan |
|---|---:|---:|---|---:|---|
| `is_promo` | int64 | 2 | 0 | 83.0% | |
| `n_hub_seg` | int64 | 1 | 1 | 100.0% | Konstan (>99%) |
| `has_scan` | int8 | 1 | 1 | 100.0% | Konstan (>99%) |
| `has_route` | int8 | 1 | 1 | 100.0% | Konstan (>99%) |
| `beda_hub_kurir` | int8 | 1 | 0 | 100.0% | Konstan (>99%) |
| `lewat_kapasitas` | int8 | 2 | 0 | 99.0% | |
| `cod_dilarang` | int8 | 1 | 0 | 100.0% | Konstan (>99%) |
| `n_upaya` | int64 | 7 | 1 | 33.9% | |
| `has_selesai` | int8 | 2 | 1 | 66.8% | |
| `last_event` | str | 5 | `upaya_antar` | 54.0% | |
| `n_scan` | int64 | 18 | 4 | 20.6% | |
| `jumlah_catatan` | int64 | 8 | 0 | 58.8% | |
| `has_note` | int8 | 2 | 0 | 58.8% | |
| `flag_alamat_baru` | int64 | 2 | 0 | 86.9% | |
| `fragility_flag` | int64 | 2 | 0 | 85.5% | |
| `n_gap_gt_48` | int8 | 5 | 0 | 98.2% | |
| `n_gap_gt_24` | int8 | 7 | 0 | 80.9% | |
| `gap_max` | float64 | 9,065 | 11.24 | 0.1% | |
| `gap_std` | float64 | 1,090,361 | NaN | 2.0% | |
| `ev_upaya_antar` | int64 | 11 | 1 | 33.9% | |
| `ev_pickup` | int64 | 10 | 1 | 55.9% | |
| `sv_cod_allowed` | int64 | 2 | 0 | 56.5% | |
| `sv_max_weight_kg` | int64 | 5 | 5 | 24.2% | |
| `kr_hub_id` | str | 654 | `HUB-HK38UG82` | 0.2% | |

### Distribusi tipe event

Distribusi `event_type` pada `scan_events` train:

| Event type | Jumlah |
|---|---:|
| `pickup` | 1,970,057 |
| `upaya_antar` | 1,855,144 |
| `selesai` | 831,294 |
| `keluar_hub` | 727,125 |
| `sortasi` | 727,051 |
| `transit` | 726,440 |

- Shipment tanpa `selesai`: **412,323**
- Total shipment unik: **1,243,617**

### Pemeriksaan fitur EDA-strong: distribusi per kelas

#### `jumlah_catatan`

| Kelas | Mean | Std | Median |
|---:|---:|---:|---:|
| 0 | 0.4915 | 0.6370 | 0.0 |
| 1 | 0.3813 | 0.5998 | 0.0 |
| 2 | 0.7685 | 0.6503 | 1.0 |
| 3 | 0.3573 | 0.5847 | 0.0 |
| 4 | 0.4628 | 0.6251 | 0.0 |
| 5 | 0.3684 | 0.5891 | 0.0 |
| 6 | 0.4075 | 0.6087 | 0.0 |

#### `flag_alamat_baru`

| Kelas | Mean | Std | Median |
|---:|---:|---:|---:|
| 0 | 0.1428 | 0.3498 | 0.0 |
| 1 | 0.1090 | 0.3117 | 0.0 |
| 2 | 0.1984 | 0.3988 | 0.0 |
| 3 | 0.1000 | 0.3000 | 0.0 |
| 4 | 0.1346 | 0.3413 | 0.0 |
| 5 | 0.1077 | 0.3100 | 0.0 |
| 6 | 0.1141 | 0.3180 | 0.0 |

#### `fragility_flag`

| Kelas | Mean | Std | Median |
|---:|---:|---:|---:|
| 0 | 0.1464 | 0.3535 | 0.0 |
| 1 | 0.1461 | 0.3532 | 0.0 |
| 2 | 0.1455 | 0.3526 | 0.0 |
| 3 | 0.1453 | 0.3524 | 0.0 |
| 4 | 0.1472 | 0.3543 | 0.0 |
| 5 | 0.1462 | 0.3533 | 0.0 |
| 6 | 0.1431 | 0.3502 | 0.0 |

#### `n_upaya`

| Kelas | Mean | Std | Median |
|---:|---:|---:|---:|
| 0 | 1.5805 | 1.2183 | 1.0 |
| 1 | 1.3509 | 1.1504 | 1.0 |
| 2 | 1.8823 | 1.3160 | 2.0 |
| 3 | 1.3079 | 1.1331 | 1.0 |
| 4 | 1.5120 | 1.2011 | 1.0 |
| 5 | 1.3401 | 1.1418 | 1.0 |
| 6 | 1.3884 | 1.1630 | 1.0 |

#### `has_selesai`

| Kelas | Mean | Std | Median |
|---:|---:|---:|---:|
| 0 | 0.6482 | 0.4775 | 1.0 |
| 1 | 0.6999 | 0.4583 | 1.0 |
| 2 | 0.5818 | 0.4933 | 1.0 |
| 3 | 0.7092 | 0.4541 | 1.0 |
| 4 | 0.6629 | 0.4727 | 1.0 |
| 5 | 0.7014 | 0.4576 | 1.0 |
| 6 | 0.6906 | 0.4623 | 1.0 |


## 7. Membuat submission

In [22]:
# pred_test = np.argmax(test_avg * best_w, axis=1)
#
# sub = pd.DataFrame({
#     "shipment_id": test_ids,
#     TARGET: classes[pred_test]
# })
#
# submission_path = BASE / "submission.csv"
# sub.to_csv(submission_path, index=False)
#
# print("Submission:", sub.shape)
# print("Saved to:", submission_path)
# print(sub[TARGET].value_counts(normalize=True).round(3))
